In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df1 = pd.read_csv("/content/drive/MyDrive/Colab/myvietnam/df1_tachtu_xuly.csv")

In [ ]:
df2 = pd.read_csv('/content/drive/MyDrive/Colab/myvietnam/df2_lai_tachtu.csv')
df3 = pd.read_csv('/content/drive/MyDrive/Colab/myvietnam/df3_lai_tachtu.csv')
df4 = pd.read_csv('/content/drive/MyDrive/Colab/myvietnam/df4_lai_tachtu.csv')
df5 = pd.read_csv('/content/drive/MyDrive/Colab/myvietnam/df5_lai_tachtu.csv')

In [ ]:
df5

,title,label,title_segmented
0,"""Trùm"" logistics Sotrans lên kế hoạch tăng gấp...",2,""" Trùm "" logistics Sotrans lên kế_hoạch tăng g..."
1,"ĐHĐCĐ SSI: ""Khen ngợi CTCK thị phần số 1, nhưn...",1,"ĐHĐCĐ SSI : "" Khen_ngợi CTCK thị_phần số 1 , n..."
2,Cổ phiếu nhóm Viettel giảm mạnh dù thị trường ...,1,Cổ_phiếu nhóm Viettel giảm mạnh dù thị_trường ...
3,"Cổ phiếu lên đỉnh, con gái chủ tịch Haxaco (HA...",1,"Cổ_phiếu lên đỉnh , con gái chủ_tịch Haxaco ( ..."
4,"FPT: Cổ phiếu bứt phá 59%, lãi ròng 4 tháng đầ...",2,"FPT : Cổ_phiếu bứt_phá 59% , lãi_ròng 4 tháng ..."
...,...,...,...
1000,"Kỳ vọng cược tàu vẫn ở mức cao, Hải An (HAH) đ...",2,"Kỳ_vọng cược tàu vẫn ở mức cao , Hải_An ( HAH ..."
1001,Giá dầu tăng mạnh tác động ra sao tới cổ phiếu...,1,Giá dầu tăng mạnh tác_động ra sao tới cổ_phiếu...
1002,"Lãnh đạo yếu kém, mâu thuẫn nội bộ khiến cổ đô...",0,"Lãnh_đạo yếu_kém , mâu_thuẫn nội_bộ khiến cổ_đ..."
1003,VCSC đón nhận chuỗi giải thưởng quốc tế danh giá,2,VCSC đón_nhận chuỗi giải_thưởng quốc_tế danh_giá


In [ ]:
# STEP 2 — CURRICULUM LEARNING (LENGTH ORDER) — PhoBERT CLEAN

!pip install -q transformers datasets scikit-learn

import os
import gc
import shutil
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

# 0) REPRODUCIBILITY
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1) HELPERS
def ensure_empty_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def safe_empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 2) PATHS & FILES
C0_MODEL = "/content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2"

DATA_FILES = {
    "task1": "/content/drive/MyDrive/Colab/myvietnam/df1_tachtu_xuly.csv",
    "task2": "/content/drive/MyDrive/Colab/myvietnam/df2_lai_tachtu.csv",
    "task3": "/content/drive/MyDrive/Colab/myvietnam/df3_lai_tachtu.csv",
    "task4": "/content/drive/MyDrive/Colab/myvietnam/df4_lai_tachtu.csv",
}

OUTPUT_BASE = "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_length"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# 3) TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(C0_MODEL, use_fast=False)
MAX_LEN = 256

# 4) LOAD DATA
df1 = pd.read_csv(DATA_FILES["task1"])
df2 = pd.read_csv(DATA_FILES["task2"])
df3 = pd.read_csv(DATA_FILES["task3"])
df4 = pd.read_csv(DATA_FILES["task4"])

# 5) CHECK REQUIRED COLUMNS
required_cols = {
    "task1": ["text_segmented"],
    "task2": ["title_segmented"],
    "task3": ["title_segmented", "cluster_label"],
    "task4": ["text_segmented", "label"],
}

for task_name, cols in required_cols.items():
    df = {"task1": df1, "task2": df2, "task3": df3, "task4": df4}[task_name]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{task_name} is missing columns: {missing}")

# 6) NORMALIZE TEXT COLUMNS
df1["text"] = df1["text_segmented"].fillna("").astype(str)
df2["text"] = df2["title_segmented"].fillna("").astype(str)
df3["text"] = df3["title_segmented"].fillna("").astype(str)
df4["text"] = df4["text_segmented"].fillna("").astype(str)

df1 = df1[df1["text"].str.strip() != ""].reset_index(drop=True)
df2 = df2[df2["text"].str.strip() != ""].reset_index(drop=True)
df3 = df3[df3["text"].str.strip() != ""].reset_index(drop=True)
df4 = df4[df4["text"].str.strip() != ""].reset_index(drop=True)

# 7) FIX LABELS
# task3: remap into 0.49
df3["cluster_label"] = df3["cluster_label"].astype("category").cat.codes

print("\nFIXED task3 labels:")
print("Unique =", df3["cluster_label"].nunique())
print("Min =", df3["cluster_label"].min(), "Max =", df3["cluster_label"].max())

assert df3["cluster_label"].min() == 0, "task3 min label must be 0"
assert df3["cluster_label"].max() == 49, "task3 max label must be 49"
assert df3["cluster_label"].nunique() == 50, "task3 must have exactly 50 labels"

# task4
df4["label"] = df4["label"].astype(int)

print("\n🔧 task4 labels:")
print("Unique =", sorted(df4["label"].unique()))
print("Min =", df4["label"].min(), "Max =", df4["label"].max())

# 8) LENGTH-BASED CURRICULUM SCORE
def compute_avg_len(texts):
    lengths = [len(tokenizer.encode(str(t), add_special_tokens=True)) for t in texts]
    return float(np.mean(lengths))

length_scores = {
    "task1": compute_avg_len(df1["text"]),
    "task2": compute_avg_len(df2["text"]),
    "task3": compute_avg_len(df3["text"]),
    "task4": compute_avg_len(df4["text"]),
}

print("\LENGTH SCORES:")
for k, v in length_scores.items():
    print(f"{k}: {v:.2f}")

CURRICULUM_ORDER = sorted(length_scores, key=length_scores.get)
print("\n Curriculum order (LENGTH):", CURRICULUM_ORDER)

# 9) DATASET CONVERSION

def to_dataset(df, text_col="text", label_col=None):
    if label_col is None:
        ds = Dataset.from_pandas(df[[text_col]].copy(), preserve_index=False)
    else:
        ds = Dataset.from_pandas(df[[text_col, label_col]].copy(), preserve_index=False)
        ds = ds.rename_column(label_col, "labels")

    ds = ds.map(
        lambda batch: tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        ),
        batched=True
    )
    return ds

# 10) TRAINING FUNCTIONS
def train_unsupervised_MLM(model_path, df, save_path):
    ensure_empty_dir(save_path)

    train_df, val_df = train_test_split(
        df,
        test_size=0.1,
        random_state=SEED,
        shuffle=True
    )

    train_ds = to_dataset(train_df, text_col="text")
    val_ds   = to_dataset(val_df, text_col="text")

    model = AutoModelForMaskedLM.from_pretrained(model_path)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=2,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=3e-5,
            fp16=torch.cuda.is_available(),
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm_probability=0.15
        ),
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f" Saved MLM model at: {save_path}")

    del trainer, model
    safe_empty_cache()
    return save_path


def train_supervised(model_path, df, label_col, save_path, num_labels):
    ensure_empty_dir(save_path)

    train_df, val_df = train_test_split(
        df,
        test_size=0.1,
        random_state=SEED,
        stratify=df[label_col],
        shuffle=True
    )

    train_ds = to_dataset(train_df, text_col="text", label_col=label_col)
    val_ds   = to_dataset(val_df, text_col="text", label_col=label_col)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=3e-5,
            fp16=torch.cuda.is_available(),
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f" Saved supervised model at: {save_path}")

    del trainer, model
    safe_empty_cache()
    return save_path

# 11) RUN LENGTH CURRICULUM
current_model = C0_MODEL

for task in CURRICULUM_ORDER:
    print(f" TRAINING {task} (LENGTH curriculum)")

    if task == "task1":
        current_model = train_unsupervised_MLM(
            current_model,
            df1,
            f"{OUTPUT_BASE}/C1_task1"
        )

    elif task == "task2":
        current_model = train_unsupervised_MLM(
            current_model,
            df2,
            f"{OUTPUT_BASE}/C2_task2"
        )

    elif task == "task3":
        current_model = train_supervised(
            current_model,
            df3,
            "cluster_label",
            f"{OUTPUT_BASE}/C3_task3",
            num_labels=50
        )

    elif task == "task4":
        current_model = train_supervised(
            current_model,
            df4,
            "label",
            f"{OUTPUT_BASE}/C4_task4",
            num_labels=3
        )

print("\n CURRICULUM (LENGTH) TRAINING COMPLETE!")
print(" Final model at:", current_model)
print(" Length scores:", length_scores)
print(" Curriculum order:", CURRICULUM_ORDER)


🔧 FIXED task3 labels:
Unique = 50
Min = 0 Max = 49

🔧 task4 labels:
Unique = [np.int64(0), np.int64(1), np.int64(2)]
Min = 0 Max = 2

🔍 LENGTH SCORES:
task1: 50.12
task2: 13.96
task3: 14.54
task4: 13.40

 Curriculum order (LENGTH): ['task4', 'task2', 'task3', 'task1']

 TRAINING task4 (LENGTH curriculum)


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss
1,No log,6.064524
2,No log,5.781803


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Saved MLM model at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_length/C1_task1

 CURRICULUM (LENGTH) TRAINING COMPLETE!
 Final model at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_length/C1_task1
 Length scores: {'task1': 50.12025769506084, 'task2': 13.9575, 'task3': 14.541666666666666, 'task4': 13.4045}
 Curriculum order: ['task4', 'task2', 'task3', 'task1']


In [ ]:
# STEP 2 — CURRICULUM LEARNING (MODEL-BASED CE ORDER) — PhoBERT

!pip install transformers datasets scikit-learn -q

import os, gc, math, random, shutil
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed,
)

# GLOBAL DETERMINISM

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def reset_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

def ensure_empty_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

# PATHS


C0_MODEL = "/content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2"

DATA_FILES = {
    "task1": "/content/drive/MyDrive/Colab/myvietnam/df1_tachtu_xuly.csv",
    "task2": "/content/drive/MyDrive/Colab/myvietnam/df2_lai_tachtu.csv",
    "task3": "/content/drive/MyDrive/Colab/myvietnam/df3_lai_tachtu.csv",
    "task4": "/content/drive/MyDrive/Colab/myvietnam/df4_lai_tachtu.csv",
}

OUTPUT_BASE = "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_CE_model_based"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(C0_MODEL, use_fast=False)
MAX_LEN = 256

# PhoBERT / RoBERTa-style models usually use <mask>
MASK_TOKEN_ID = tokenizer.mask_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id
CLS_TOKEN_ID = tokenizer.cls_token_id
SEP_TOKEN_ID = tokenizer.sep_token_id

SPECIAL_TOKEN_IDS = set(
    tid for tid in [PAD_TOKEN_ID, CLS_TOKEN_ID, SEP_TOKEN_ID]
    if tid is not None
)

# LOAD DATA

df1 = pd.read_csv(DATA_FILES["task1"])
df2 = pd.read_csv(DATA_FILES["task2"])
df3 = pd.read_csv(DATA_FILES["task3"])
df4 = pd.read_csv(DATA_FILES["task4"])

df1["text"] = df1["text_segmented"].astype(str)
df2["text"] = df2["title_segmented"].astype(str)
df3["text"] = df3["title_segmented"].astype(str)
df4["text"] = df4["text_segmented"].astype(str)

# Remap task3 labels to 0-49 for stable classifier head
df3["cluster_label"] = df3["cluster_label"].astype("category").cat.codes
assert df3["cluster_label"].min() == 0
assert df3["cluster_label"].max() == 49

# DATASET HELPERS

def to_dataset(df, text_col="text", label_col=None):
    if label_col is None:
        ds = Dataset.from_pandas(df[[text_col]])
    else:
        ds = Dataset.from_pandas(df[[text_col, label_col]]).rename_column(label_col, "labels")

    ds = ds.map(
        lambda batch: tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        ),
        batched=True
    )
    return ds

# DETERMINISTIC MLM MASKING

def build_deterministic_mlm_example(text, example_idx, mlm_probability=0.15):
    """
    Create one fixed masked example for MLM scoring.
    This avoids random masking from DataCollatorForLanguageModeling.
    """
    enc = tokenizer(
        str(text),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_attention_mask=True
    )

    input_ids = enc["input_ids"][:]
    attention_mask = enc["attention_mask"][:]
    labels = [-100] * len(input_ids)

    candidate_positions = []
    for pos, tid in enumerate(input_ids):
        if attention_mask[pos] != 1:
            continue
        if tid in SPECIAL_TOKEN_IDS:
            continue
        candidate_positions.append(pos)

    if len(candidate_positions) == 0:
        return {
            "input_ids": torch.tensor([input_ids], dtype=torch.long),
            "attention_mask": torch.tensor([attention_mask], dtype=torch.long),
            "labels": torch.tensor([labels], dtype=torch.long),
        }

    num_to_mask = max(1, int(round(len(candidate_positions) * mlm_probability)))

    # Deterministic local RNG per example
    local_rng = random.Random(SEED + int(example_idx))
    masked_positions = sorted(local_rng.sample(candidate_positions, min(num_to_mask, len(candidate_positions))))

    for pos in masked_positions:
        labels[pos] = input_ids[pos]
        input_ids[pos] = MASK_TOKEN_ID

    return {
        "input_ids": torch.tensor([input_ids], dtype=torch.long),
        "attention_mask": torch.tensor([attention_mask], dtype=torch.long),
        "labels": torch.tensor([labels], dtype=torch.long),
    }

#  MODEL-BASED SCORE FUNCTIONS

def compute_model_based_mlm_loss(model_path, df, text_col="text", max_samples=800):
    """
    Mean deterministic MLM loss of C0 on one task.
    """
    reset_all_seeds(SEED)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE)
    model.eval()

    texts = df[text_col].astype(str).tolist()[:max_samples]
    losses = []

    with torch.no_grad():
        for i, txt in enumerate(texts):
            batch = build_deterministic_mlm_example(txt, example_idx=i, mlm_probability=0.15)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            losses.append(float(outputs.loss.item()))

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return float(np.mean(losses))

def compute_model_based_cls_loss(model_path, df, text_col, label_col, num_labels, max_samples=None):
    """
    Mean classification CE loss of C0 on one supervised task.
    Fresh classifier head is initialized once, then evaluated over the task.
    """
    reset_all_seeds(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    ).to(DEVICE)
    model.eval()

    if max_samples is not None:
        df = df.iloc[:max_samples].copy()

    texts = df[text_col].astype(str).tolist()
    labels = df[label_col].tolist()

    losses = []

    with torch.no_grad():
        for txt, y in zip(texts, labels):
            enc = tokenizer(
                str(txt),
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt"
            )
            batch = {
                "input_ids": enc["input_ids"].to(DEVICE),
                "attention_mask": enc["attention_mask"].to(DEVICE),
                "labels": torch.tensor([int(y)], dtype=torch.long, device=DEVICE)
            }
            outputs = model(**batch)
            losses.append(float(outputs.loss.item()))

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return float(np.mean(losses))

# COMPUTE MODEL-BASED CE SCORES

ce_scores = {
    "task1": compute_model_based_mlm_loss(C0_MODEL, df1, text_col="text", max_samples=800),
    "task2": compute_model_based_mlm_loss(C0_MODEL, df2, text_col="text", max_samples=800),
    "task3": compute_model_based_cls_loss(C0_MODEL, df3, text_col="text", label_col="cluster_label", num_labels=50),
    "task4": compute_model_based_cls_loss(C0_MODEL, df4, text_col="text", label_col="label", num_labels=3),
}

print("\nMODEL-BASED CE SCORES:")
for k, v in ce_scores.items():
    print(f"{k}: {v:.6f}")

CURRICULUM_ORDER = sorted(ce_scores, key=ce_scores.get)
print("\nCurriculum order (MODEL-BASED CE):", CURRICULUM_ORDER)

# TRAINING FUNCTIONS

def train_unsupervised_MLM(model_path, df, save_path):
    ensure_empty_dir(save_path)
    reset_all_seeds(SEED)

    train_df, val_df = train_test_split(df, test_size=0.1, random_state=SEED)

    train_ds = to_dataset(train_df)
    val_ds   = to_dataset(val_df)

    model = AutoModelForMaskedLM.from_pretrained(model_path)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=2,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            eval_strategy="epoch",
            learning_rate=3e-5,
            fp16=False,                  # for stability
            dataloader_num_workers=0,    # for stability
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm_probability=0.15
        )
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved MLM model at {save_path}")

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return save_path

def train_supervised(model_path, df, label_col, save_path, num_labels):
    ensure_empty_dir(save_path)
    reset_all_seeds(SEED)

    train_df, val_df = train_test_split(
        df,
        test_size=0.1,
        random_state=SEED,
        stratify=df[label_col]
    )

    train_ds = to_dataset(train_df, label_col=label_col)
    val_ds   = to_dataset(val_df, label_col=label_col)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy="epoch",
            learning_rate=3e-5,
            fp16=False,                  # for stability
            dataloader_num_workers=0,    # for stability
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved supervised model at {save_path}")

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return save_path

# RUN CURRICULUM

current_model = C0_MODEL

for task in CURRICULUM_ORDER:
    print("\n==============================")
    print(f"TRAINING {task} (according to MODEL-BASED CE)")
    print("==============================")

    if task == "task1":
        current_model = train_unsupervised_MLM(
            current_model, df1, f"{OUTPUT_BASE}/C1_task1"
        )

    elif task == "task2":
        current_model = train_unsupervised_MLM(
            current_model, df2, f"{OUTPUT_BASE}/C2_task2"
        )

    elif task == "task3":
        current_model = train_supervised(
            current_model, df3, "cluster_label",
            f"{OUTPUT_BASE}/C3_task3", num_labels=50
        )

    elif task == "task4":
        current_model = train_supervised(
            current_model, df4, "label",
            f"{OUTPUT_BASE}/C4_task4", num_labels=3
        )

print("\n CURRICULUM (MODEL-BASED CE) TRAINING COMPLETE!")
print(" Final model at:", current_model)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_CE_model_based/C2_task2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,3.073108
2,3.187510,2.714835
3,3.187510,2.620075


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🔥 Saved supervised model at /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_CE_model_based/C3_task3

🎉 CURRICULUM (MODEL-BASED CE) TRAINING COMPLETE!
📌 Final model at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_CE_model_based/C3_task3


In [ ]:
# STEP 2 — CURRICULUM LEARNING (WARMUP / OPTIMIZATION-LEVEL) — PhoBERT

!pip install transformers datasets scikit-learn -q

import os, gc, random, shutil
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

# GLOBAL DETERMINISM

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def reset_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

def ensure_empty_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

# PATHS

C0_MODEL = "/content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2"

DATA_FILES = {
    "task1": "/content/drive/MyDrive/Colab/myvietnam/df1_tachtu_xuly.csv",
    "task2": "/content/drive/MyDrive/Colab/myvietnam/df2_lai_tachtu.csv",
    "task3": "/content/drive/MyDrive/Colab/myvietnam/df3_lai_tachtu.csv",
    "task4": "/content/drive/MyDrive/Colab/myvietnam/df4_lai_tachtu.csv",
}

OUTPUT_BASE = "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_warmup_opt"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# TOKENIZER / CONFIG

tokenizer = AutoTokenizer.from_pretrained(C0_MODEL, use_fast=False)
MAX_LEN = 256
WARMUP_RATIO = 0.10
LR = 3e-5

MASK_TOKEN_ID = tokenizer.mask_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id
CLS_TOKEN_ID = tokenizer.cls_token_id
SEP_TOKEN_ID = tokenizer.sep_token_id

SPECIAL_TOKEN_IDS = {
    tid for tid in [PAD_TOKEN_ID, CLS_TOKEN_ID, SEP_TOKEN_ID]
    if tid is not None
}

# LOAD DATA

df1 = pd.read_csv(DATA_FILES["task1"])
df2 = pd.read_csv(DATA_FILES["task2"])
df3 = pd.read_csv(DATA_FILES["task3"])
df4 = pd.read_csv(DATA_FILES["task4"])

df1["text"] = df1["text_segmented"].astype(str)
df2["text"] = df2["title_segmented"].astype(str)
df3["text"] = df3["title_segmented"].astype(str)
df4["text"] = df4["text_segmented"].astype(str)

# Remap task3 labels to sequential 0-49
df3["cluster_label"] = df3["cluster_label"].astype("category").cat.codes
assert df3["cluster_label"].min() == 0
assert df3["cluster_label"].max() == 49

#  4) FIXED TASK SEQUENCE

TASK_SEQUENCE = ["task1", "task2", "task3", "task4"]

print("Warmup task sequence:", TASK_SEQUENCE)

#  DATASET HELPERS

def to_dataset_supervised(df, text_col="text", label_col=None):
    if label_col is None:
        raise ValueError("label_col must be provided for supervised training.")

    ds = Dataset.from_pandas(df[[text_col, label_col]]).rename_column(label_col, "labels")
    ds = ds.map(
        lambda batch: tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        ),
        batched=True
    )
    return ds

def build_fixed_mlm_dataset(df, text_col="text", mlm_probability=0.15):
    """
    Build deterministic masked MLM dataset:
    same example index -> same masked positions across runs.
    """
    texts = df[text_col].astype(str).tolist()
    records = []

    for idx, txt in enumerate(texts):
        enc = tokenizer(
            txt,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_attention_mask=True
        )

        input_ids = enc["input_ids"][:]
        attention_mask = enc["attention_mask"][:]
        labels = [-100] * len(input_ids)

        candidate_positions = []
        for pos, tid in enumerate(input_ids):
            if attention_mask[pos] != 1:
                continue
            if tid in SPECIAL_TOKEN_IDS:
                continue
            candidate_positions.append(pos)

        if candidate_positions:
            num_to_mask = max(1, int(round(len(candidate_positions) * mlm_probability)))
            local_rng = random.Random(SEED + idx)
            masked_positions = sorted(
                local_rng.sample(candidate_positions, min(num_to_mask, len(candidate_positions)))
            )

            for pos in masked_positions:
                labels[pos] = input_ids[pos]
                input_ids[pos] = MASK_TOKEN_ID

        records.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        })

    return Dataset.from_pandas(pd.DataFrame(records))

# TRAINING FUNCTIONS WITH OPTIMIZATION-LEVEL WARMUP

def train_unsupervised_MLM_warmup(model_path, df, save_path):
    ensure_empty_dir(save_path)
    reset_all_seeds(SEED)

    train_df, val_df = train_test_split(df, test_size=0.1, random_state=SEED)

    train_ds = build_fixed_mlm_dataset(train_df, text_col="text", mlm_probability=0.15)
    val_ds   = build_fixed_mlm_dataset(val_df,   text_col="text", mlm_probability=0.15)

    model = AutoModelForMaskedLM.from_pretrained(model_path)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=2,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            eval_strategy="epoch",
            learning_rate=LR,
            warmup_ratio=WARMUP_RATIO,     # warmup at optimization level
            lr_scheduler_type="linear",
            fp16=False,                    # stability
            dataloader_num_workers=0,      # stability
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved MLM warmup model at {save_path}")

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return save_path


def train_supervised_warmup(model_path, df, label_col, save_path, num_labels):
    ensure_empty_dir(save_path)
    reset_all_seeds(SEED)

    train_df, val_df = train_test_split(
        df,
        test_size=0.1,
        random_state=SEED,
        stratify=df[label_col]
    )

    train_ds = to_dataset_supervised(train_df, text_col="text", label_col=label_col)
    val_ds   = to_dataset_supervised(val_df,   text_col="text", label_col=label_col)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=save_path,
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy="epoch",
            learning_rate=LR,
            warmup_ratio=WARMUP_RATIO,     # warmup at optimization level
            lr_scheduler_type="linear",
            fp16=False,                    # stability
            dataloader_num_workers=0,      # stability
            save_total_limit=1,
            logging_steps=200,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved supervised warmup model at {save_path}")

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return save_path

# ------------------------------------------------------------
# 7) RUN STEP 2 — WARMUP / OPTIMIZATION-LEVEL
# ------------------------------------------------------------

current_model = C0_MODEL

for task in TASK_SEQUENCE:
    print(f"TRAINING {task} (warmup optimization-level)")

    if task == "task1":
        current_model = train_unsupervised_MLM_warmup(
            current_model, df1, f"{OUTPUT_BASE}/C1_task1"
        )

    elif task == "task2":
        current_model = train_unsupervised_MLM_warmup(
            current_model, df2, f"{OUTPUT_BASE}/C2_task2"
        )

    elif task == "task3":
        current_model = train_supervised_warmup(
            current_model, df3, "cluster_label",
            f"{OUTPUT_BASE}/C3_task3", num_labels=50
        )

    elif task == "task4":
        current_model = train_supervised_warmup(
            current_model, df4, "label",
            f"{OUTPUT_BASE}/C4_task4", num_labels=3
        )

print("\n CURRICULUM (WARMUP / OPTIMIZATION-LEVEL) COMPLETE!")
print(" Final model at:", current_model)

🔥 Warmup task sequence: ['task1', 'task2', 'task3', 'task4']

🚀 TRAINING task1 (warmup optimization-level)


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,No log,1.471946
2,No log,1.459823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_warmup_opt/C3_task3
Key                        | Status   |                                                                                      
---------------------------+----------+--------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([50, 768]) vs model:torch.Size([3, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([50]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,No log,0.326259
2,0.392626,0.286939
3,0.392626,0.321455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🔥 Saved supervised warmup model at /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_warmup_opt/C4_task4

🎉 CURRICULUM (WARMUP / OPTIMIZATION-LEVEL) COMPLETE!
📌 Final model at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_warmup_opt/C4_task4


In [ ]:
# STEP 3 — LSDS FUSION (FINAL, THESIS-MATCHED, STABILITY-ORIENTED)

!pip install -q transformers safetensors

# ENV + GLOBAL DETERMINISM
import os

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import json
import random
import numpy as np
import torch

from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM, set_seed

def make_deterministic(seed=42):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if torch.cuda.is_available():
        try:
            torch.backends.cuda.matmul.allow_tf32 = False
        except Exception:
            pass
        try:
            torch.backends.cudnn.allow_tf32 = False
        except Exception:
            pass

    try:
        torch.use_deterministic_algorithms(True, warn_only=False)
    except TypeError:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Deterministic algorithms not fully supported:", e)

make_deterministic(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

# 1) PATHS
BASE_ARCH = "/content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2"

# KEEP EXACT USER PATHS
paths = {
    "C0": "/content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2",
    "C1": "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_length/C1_task1",
    "C2": "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_CE_model_based/C3_task3",
    "C3": "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_warmup_opt/C4_task4",
}

DEV_TEXT = "/content/drive/MyDrive/Colab/myvietnam/df35_unlabelled_xuly.csv"
OUT_DIR  = "/content/drive/MyDrive/Colab/myvietnam/cuoi_output_step3_lsds_fused_final_thesis"
os.makedirs(OUT_DIR, exist_ok=True)

# 2) TOKENIZER
tokenizer_lsds = AutoTokenizer.from_pretrained(BASE_ARCH, use_fast=False)

assert tokenizer_lsds.mask_token_id is not None, "Tokenizer has no mask_token_id."

VOCAB_SIZE = len(tokenizer_lsds)
MASK_TOKEN_ID = tokenizer_lsds.mask_token_id
PAD_TOKEN_ID = tokenizer_lsds.pad_token_id

# 3) DEV MLM DATASET (DETERMINISTIC MASKING)
@dataclass
class MLMConfig:
    mlm_probability: float = 0.15
    max_length: int = 128
    seed: int = SEED

class MLMDataset(Dataset):
    def __init__(self, tokenizer, texts, cfg: MLMConfig):
        self.tok = tokenizer
        self.texts = [str(t).strip() for t in texts if str(t).strip()]
        self.cfg = cfg

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            truncation=True,
            padding="max_length",
            max_length=self.cfg.max_length,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"][0].clone()
        attention_mask = enc["attention_mask"][0].clone()
        labels = input_ids.clone()

        # deterministic generator per example
        gen = torch.Generator(device="cpu")
        gen.manual_seed(self.cfg.seed + int(i))

        prob = torch.full(labels.shape, self.cfg.mlm_probability, dtype=torch.float32)

        special = torch.tensor(
            self.tok.get_special_tokens_mask(
                input_ids.tolist(),
                already_has_special_tokens=True
            ),
            dtype=torch.bool
        )
        prob.masked_fill_(special, 0.0)

        if PAD_TOKEN_ID is not None:
            prob.masked_fill_(input_ids.eq(PAD_TOKEN_ID), 0.0)

        masked = torch.bernoulli(prob, generator=gen).bool()

        # ensure at least one masked token if possible
        if not masked.any():
            valid_positions = torch.where((attention_mask == 1) & (~special))[0]
            if len(valid_positions) > 0:
                masked[valid_positions[0]] = True

        labels[~masked] = -100

        # 80% -> [MASK]
        replace_prob = torch.full(labels.shape, 0.8, dtype=torch.float32)
        rep = masked & torch.bernoulli(replace_prob, generator=gen).bool()
        input_ids[rep] = MASK_TOKEN_ID

        # 10% -> random token
        random_prob = torch.full(labels.shape, 0.5, dtype=torch.float32)
        rnd = masked & (~rep) & torch.bernoulli(random_prob, generator=gen).bool()
        if rnd.any():
            rand_ids = torch.randint(
                low=0,
                high=VOCAB_SIZE,
                size=labels.shape,
                generator=gen,
                dtype=torch.long
            )
            input_ids[rnd] = rand_ids[rnd]

        # remaining 10% keep original
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

def collate(batch):
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0]}

# 4) LOAD CHECKPOINT STATES
def load_state(folder):
    """
    Load full state dict via from_pretrained for compatibility with
    both safetensors and pytorch_model.bin.
    """
    print(f"Loading checkpoint from: {folder}")
    model = AutoModelForMaskedLM.from_pretrained(folder)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return state

# 5) FUSION UTILITIES
def fuse_state_dicts(states, weights):
    """
    Weighted fusion of compatible tensors across checkpoints.
    If shape mismatch occurs, fallback to first state.
    """
    fused = {}
    ref_state = states[0]

    for k in ref_state.keys():
        compatible = all((k in s) and (s[k].shape == ref_state[k].shape) for s in states)

        if compatible:
            stacked = torch.stack([s[k].float() for s in states], dim=0)
            view_shape = (-1,) + (1,) * (stacked.dim() - 1)
            w = weights.view(*view_shape)
            fused[k] = (stacked * w).sum(dim=0)
        else:
            fused[k] = ref_state[k].clone()

    return fused

def project_simplex(w):
    """
    Project weight vector onto probability simplex:
    nonnegative and sum to 1.
    """
    w = torch.clamp(w, min=0)
    s = w.sum()
    if s.item() <= 0:
        return torch.ones_like(w) / w.numel()
    return w / s

# 6) DEV EVALUATOR
class DevEvaluator:
    def __init__(self, base_arch, device, loader):
        self.model = AutoModelForMaskedLM.from_pretrained(base_arch).to(device)
        self.model.eval()
        self.device = device
        self.loader = loader

    @torch.no_grad()
    def eval_loss(self, fused_sd):
        self.model.load_state_dict(fused_sd, strict=False)
        self.model.eval()

        total_loss = 0.0
        total_n = 0

        for batch in self.loader:
            batch = {k: v.to(self.device, non_blocking=False) for k, v in batch.items()}
            outputs = self.model(**batch)

            bs = batch["input_ids"].size(0)
            total_loss += float(outputs.loss.item()) * bs
            total_n += bs

        return total_loss / max(total_n, 1)

# 7) LSDS SEARCH
def lsds_search(states, evaluator, steps=(0.50, 0.10, 0.05, 0.01), tol=1e-12):
    """
    Local Search with Decreasing Size:
    - start from equal weights
    - perturb each coordinate +/- step
    - project to simplex
    - keep strict improvements
    - reduce step size gradually
    """
    M = len(states)
    w = torch.ones(M, dtype=torch.float32) / M

    best_loss = evaluator.eval_loss(fuse_state_dicts(states, w))
    best_w = w.clone()

    print(f"Initial weights: {best_w.tolist()}")
    print(f"Initial dev loss: {best_loss:.10f}")

    for step in steps:
        print(f"\nLSDS step size = {step}")
        improved = True

        while improved:
            improved = False

            for i in range(M):
                for sign in (+1, -1):
                    w_try = w.clone()
                    w_try[i] += sign * step
                    w_try = project_simplex(w_try)

                    fused_try = fuse_state_dicts(states, w_try)
                    loss_try = evaluator.eval_loss(fused_try)

                    sign_txt = f"+{step:.2f}" if sign > 0 else f"-{step:.2f}"
                    print(
                        f"   → dim={i}, delta={sign_txt}, "
                        f"loss={loss_try:.10f}, weights={w_try.tolist()}"
                    )

                    if loss_try < best_loss - tol:
                        best_loss = loss_try
                        w = w_try.clone()
                        best_w = w_try.clone()
                        improved = True
                        print(f" improved -> best_loss={best_loss:.10f}")

                    del fused_try
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

    return best_w, best_loss

# 8) BUILD DEV LOADER
make_deterministic(SEED)

with open(DEV_TEXT, "r", encoding="utf-8") as f:
    dev_texts = [line.strip() for line in f if line.strip()]

# fixed subset for reproducibility
dev_texts = dev_texts[:1000]

dev_dataset = MLMDataset(
    tokenizer=tokenizer_lsds,
    texts=dev_texts,
    cfg=MLMConfig(mlm_probability=0.15, max_length=128, seed=SEED)
)

loader = DataLoader(
    dev_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate,
    num_workers=0,
    pin_memory=False,
    drop_last=False
)

# 9) RUN LSDS FUSION
make_deterministic(SEED)

ordered_keys = ["C0", "C1", "C2", "C3"]
states = [load_state(paths[k]) for k in ordered_keys]

evaluator = DevEvaluator(BASE_ARCH, DEVICE, loader)
best_w, best_loss = lsds_search(states, evaluator)

# 10) OPTIONAL SELF-CHECK FOR SAME-RUN STABILITY
# re-evaluate best fused weights one more time
fused_sd_check = fuse_state_dicts(states, best_w)
check_loss = evaluator.eval_loss(fused_sd_check)

print(f"\nRecheck best dev loss: {check_loss:.10f}")
print(f"Difference from best:  {abs(check_loss - best_loss):.12f}")

# ------------------------------------------------------------
# 11) SAVE FINAL FUSED MODEL
# ------------------------------------------------------------
fused_sd = fuse_state_dicts(states, best_w)

model_fused = AutoModelForMaskedLM.from_pretrained(BASE_ARCH)
model_fused.load_state_dict(fused_sd, strict=False)
model_fused.tie_weights()
model_fused.eval()

model_fused.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer_lsds.save_pretrained(OUT_DIR)

np.save(os.path.join(OUT_DIR, "best_weights.npy"), best_w.cpu().numpy())

report = {
    "step": "STEP 3 — LSDS Fusion Report",
    "seed": SEED,
    "device": DEVICE,
    "dev_text": DEV_TEXT,
    "base_arch": BASE_ARCH,
    "best_dev_loss": float(best_loss),
    "recheck_dev_loss": float(check_loss),
    "abs_diff_recheck": float(abs(check_loss - best_loss)),
    "weights": {k: float(best_w[i].item()) for i, k in enumerate(ordered_keys)},
    "paths": paths,
    "notes": [
        "Original path logic preserved exactly as provided by user.",
        "Designed for maximum same-environment reproducibility.",
        "Fresh runtime recommended before execution."
    ]
}

with open(os.path.join(OUT_DIR, "fusion_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUT_DIR, "fusion_report.txt"), "w", encoding="utf-8") as f:
    f.write("STEP 3 — LSDS Fusion Report\n")
    f.write(f"SEED = {SEED}\n")
    f.write(f"DEVICE = {DEVICE}\n")
    f.write(f"DEV_TEXT = {DEV_TEXT}\n")
    f.write(f"BASE_ARCH = {BASE_ARCH}\n")
    f.write(f"Best dev loss = {best_loss:.10f}\n")
    f.write(f"Recheck dev loss = {check_loss:.10f}\n")
    f.write(f"Absolute difference = {abs(check_loss - best_loss):.12f}\n")
    f.write("Weights:\n")
    for i, k in enumerate(ordered_keys):
        f.write(f"{k}: {best_w[i].item():.10f}\n")
    f.write("Paths:\n")
    for k, v in paths.items():
        f.write(f"{k}: {v}\n")
    f.write("\nNotes:\n")
    f.write("- Original path mapping preserved exactly.\n")
    f.write("- Fresh runtime recommended.\n")
    f.write("- Same environment reproducibility target.\n")

print("\n=====================================")
print("STEP 3 LSDS Fusion complete.")
for i, k in enumerate(ordered_keys):
    print(f"   {k}: {best_w[i].item():.8f}")
print(f"Best dev loss: {best_loss:.10f}")
print(f"Recheck loss:  {check_loss:.10f}")
print(f"Fused model saved at: {OUT_DIR}")
print("=====================================")

DEVICE = cuda
Loading checkpoint from: /content/drive/MyDrive/Colab/myvietnam/C0_PhoBERT_base_v2


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading checkpoint from: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step2_CL_length/C1_task1


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Initial weights: [0.25, 0.25, 0.25, 0.25]
Initial dev loss: 5.4340309410

LSDS step size = 0.5
   → dim=0, delta=+0.50, loss=2.9057039509, weights=[0.5, 0.1666666716337204, 0.1666666716337204, 0.1666666716337204]
 improved -> best_loss=2.9057039509
   → dim=0, delta=-0.50, loss=9.3984288330, weights=[0.0, 0.3333333432674408, 0.3333333432674408, 0.3333333432674408]
   → dim=1, delta=+0.50, loss=4.2882047615, weights=[0.3333333432674408, 0.4444444477558136, 0.1111111119389534, 0.1111111119389534]
   → dim=1, delta=-0.50, loss=2.5616266975, weights=[0.5999999642372131, 0.0, 0.20000000298023224, 0.20000000298023224]
 improved -> best_loss=2.5616266975
   → dim=2, delta=+0.50, loss=4.0514518623, weights=[0.3999999761581421, 0.0, 0.46666666865348816, 0.13333334028720856]
   → dim=2, delta=-0.50, loss=2.1728005104, weights=[0.75, 0.0, 0.0, 0.2500000298023224]
 improved -> best_loss=2.1728005104
   → dim=3, delta=+0.50, loss=3.3919941120, weights=[0.5, 0.0, 0.0, 0.5]
   → dim=3, delta=-0.50, l

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STEP 3 LSDS Fusion complete.
   C0: 0.92561233
   C1: 0.07438760
   C2: 0.00000000
   C3: 0.00000000
Best dev loss: 1.9722805405
Recheck loss:  1.9722805405
Fused model saved at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step3_lsds_fused_final_thesis


Our Framework (With SCL)

In [ ]:
# STEP 3.5 — SUPERVISED CONTRASTIVE TINTING
# HARDENED VERSION — SAME LOGIC, TIGHTER REPRODUCIBILITY

!pip install -q transformers datasets scikit-learn sentencepiece pandas safetensors

# 0) GLOBAL DETERMINISM
import os

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import json
import math
import random
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    set_seed,
)

def lock_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    set_seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if torch.cuda.is_available():
        try:
            torch.backends.cuda.matmul.allow_tf32 = False
        except Exception:
            pass
        try:
            torch.backends.cudnn.allow_tf32 = False
        except Exception:
            pass

    try:
        torch.use_deterministic_algorithms(True, warn_only=False)
    except TypeError:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("⚠️ deterministic algorithms not fully supported:", e)

lock_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("SEED  :", SEED)

# 1) PATHS
BASE_DIR = "/content/drive/MyDrive/Colab/myvietnam"

MODEL_STEP3 = f"{BASE_DIR}/cuoi_output_step3_lsds_fused_final_thesis"
DATA_PATH   = f"{BASE_DIR}/df5_moi_tachtu.csv"

OUT_DIR_RAW   = f"{BASE_DIR}/cuoi_output_step35_supcon_tinting_final_raw"
OUT_DIR_CLEAN = f"{BASE_DIR}/cuoi_output_step35_supcon_tinting_final_clean"

os.makedirs(OUT_DIR_RAW, exist_ok=True)
os.makedirs(OUT_DIR_CLEAN, exist_ok=True)

# 2) CONFIG
@dataclass
class CFG:
    text_col: str = "title_segmented"
    label_col: str = "label"
    max_len: int = 256
    batch_size: int = 16

    epochs: int = 1
    lr: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    grad_clip: float = 1.0

    freeze_embeddings: bool = True
    freeze_bottom_n_layers: int = 9
    freeze_mlm_head: bool = True

    align_last_k_layers: int = 1
    proxy: str = "cls"   # "cls" | "mean" | "mix"
    mix_alpha: float = 0.5

    temperature: float = 0.10
    w_supcon: float = 1.0
    w_sp: float = 5e-4

cfg = CFG()
print(cfg)

# 3) LABEL MAP
label2id = {"positive": 0, "neutral": 1, "negative": 2}
id2label = {v: k for k, v in label2id.items()}

# 4) LOAD + SPLIT DATA
df = pd.read_csv(DATA_PATH)

required_cols = [cfg.text_col, cfg.label_col]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df[cfg.text_col] = df[cfg.text_col].fillna("").astype(str)

if df[cfg.label_col].dtype == object:
    unknown_labels = set(df[cfg.label_col].dropna().unique()) - set(label2id.keys())
    if unknown_labels:
        raise ValueError(f"Unknown labels found: {unknown_labels}")
    df[cfg.label_col] = df[cfg.label_col].map(label2id)

df[cfg.label_col] = df[cfg.label_col].astype(int)

if len(df.loc[~df[cfg.label_col].isin([0, 1, 2])]) > 0:
    raise ValueError("Found invalid label values outside {0,1,2}")

print("\nFULL DATA LABEL DISTRIBUTION")
print(df[cfg.label_col].value_counts().sort_index())

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df[cfg.label_col],
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[cfg.label_col],
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("\nSTEP 3.5 USES TRAIN ONLY")
print("Train:", len(train_df))
print("Val  :", len(val_df), "(not used)")
print("Test :", len(test_df), "(not used)")

print("\nTRAIN LABEL DISTRIBUTION")
print(train_df[cfg.label_col].value_counts().sort_index())

# 5) DATASET
class TrainOnlyLabeledDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_col: str, label_col: str):
        dfx = df[[text_col, label_col]].copy()
        dfx[text_col] = dfx[text_col].fillna("").astype(str)
        dfx = dfx[dfx[text_col].str.strip() != ""].reset_index(drop=True)

        self.texts = dfx[text_col].tolist()
        self.labels = dfx[label_col].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx], "label": int(self.labels[idx])}

tokenizer = AutoTokenizer.from_pretrained(MODEL_STEP3, use_fast=False)

def collate_fn(samples: List[Dict]) -> Dict[str, torch.Tensor]:
    texts = [x["text"] for x in samples]
    labels = torch.tensor([x["label"] for x in samples], dtype=torch.long)

    tok = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=cfg.max_len,
        return_tensors="pt"
    )
    tok["labels"] = labels
    return tok

train_dataset = TrainOnlyLabeledDataset(train_df, cfg.text_col, cfg.label_col)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    generator=g,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn
)

print("\nTrain dataset size:", len(train_dataset))

# 6) LOAD MODEL
lock_everything(SEED)

student = AutoModelForMaskedLM.from_pretrained(MODEL_STEP3).to(DEVICE)
student.train()

init_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}

# 7) FREEZE STRATEGY
def freeze_student_mlm(m: AutoModelForMaskedLM):
    base = getattr(m, m.base_model_prefix, None)
    if base is None:
        raise RuntimeError("Cannot locate base model in student.")

    if cfg.freeze_embeddings and hasattr(base, "embeddings"):
        for p in base.embeddings.parameters():
            p.requires_grad = False

    enc = getattr(base, "encoder", None)
    if enc is None or not hasattr(enc, "layer"):
        raise RuntimeError("Cannot locate encoder.layer in student.")

    n_layers = len(enc.layer)
    n_freeze = min(max(cfg.freeze_bottom_n_layers, 0), n_layers)

    for i in range(n_freeze):
        for p in enc.layer[i].parameters():
            p.requires_grad = False

    if cfg.freeze_mlm_head and hasattr(m, "lm_head"):
        for p in m.lm_head.parameters():
            p.requires_grad = False

    print(
        f"Freeze summary -> embeddings={cfg.freeze_embeddings}, "
        f"bottom_layers={n_freeze}/{n_layers}, mlm_head={cfg.freeze_mlm_head}"
    )

freeze_student_mlm(student)

trainable_params = [p for p in student.parameters() if p.requires_grad]
print("Trainable params:", sum(p.numel() for p in trainable_params))

# 8) REPRESENTATION HELPERS
def strip_labels(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v for k, v in batch.items() if k != "labels"}

def get_hidden_mean_lastk_grad(m, batch, last_k: int) -> torch.Tensor:
    batch_ = strip_labels(batch)
    out = m(**batch_, output_hidden_states=True, return_dict=True)
    hs = [h.float() for h in out.hidden_states]
    use = hs[-last_k:] if last_k > 0 else [hs[-1]]
    return torch.stack(use, dim=0).mean(dim=0)

def masked_mean_pool(h: torch.Tensor, attn: torch.Tensor) -> torch.Tensor:
    mask = attn.unsqueeze(-1).float()
    summed = (h * mask).sum(dim=1, keepdim=True)
    denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    return summed / denom

def get_proxy(h: torch.Tensor, attn: torch.Tensor) -> torch.Tensor:
    cls = h[:, 0:1, :]
    if cfg.proxy == "cls":
        return cls
    mean = masked_mean_pool(h, attn)
    if cfg.proxy == "mean":
        return mean
    return cfg.mix_alpha * cls + (1.0 - cfg.mix_alpha) * mean

def l2sp_penalty(student_model: nn.Module, init_state_cpu: Dict[str, torch.Tensor]) -> torch.Tensor:
    loss = torch.tensor(0.0, device=DEVICE)
    for name, p in student_model.named_parameters():
        if not p.requires_grad:
            continue
        if name in init_state_cpu:
            loss = loss + ((p.float() - init_state_cpu[name].to(DEVICE).float()) ** 2).mean()
    return loss

# 9) SUPERVISED CONTRASTIVE LOSS
def supervised_contrastive_loss(features: torch.Tensor, labels: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    if features.size(0) < 2:
        return torch.tensor(0.0, device=features.device)

    features = F.normalize(features, p=2, dim=-1)
    labels = labels.contiguous().view(-1, 1)

    mask = torch.eq(labels, labels.T).float().to(features.device)
    logits = torch.matmul(features, features.T) / temperature
    logits_max, _ = torch.max(logits, dim=1, keepdim=True)
    logits = logits - logits_max.detach()

    logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0), device=mask.device)
    mask = mask * logits_mask

    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-12))

    positive_count = mask.sum(dim=1)
    valid = positive_count > 0

    mean_log_prob_pos = torch.zeros_like(positive_count)
    mean_log_prob_pos[valid] = (mask[valid] * log_prob[valid]).sum(dim=1) / positive_count[valid]

    if valid.any():
        return -mean_log_prob_pos[valid].mean()
    return torch.tensor(0.0, device=features.device)

# 10) OPTIMIZER / SCHEDULER
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

total_steps = cfg.epochs * len(train_loader)
warmup_steps = int(total_steps * cfg.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("\nTRAIN CONFIG")
print("total_steps :", total_steps)
print("warmup_steps:", warmup_steps)

# 11) TRAIN
global_step = 0
final_epoch_avg_loss = None
final_epoch_avg_supcon = None
final_epoch_avg_sp = None

for ep in range(cfg.epochs):
    student.train()

    run_loss = 0.0
    run_supcon = 0.0
    run_sp = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE, non_blocking=False) for k, v in batch.items()}
        labels = batch["labels"]
        attn = batch["attention_mask"]

        optimizer.zero_grad(set_to_none=True)

        s_h = get_hidden_mean_lastk_grad(student, batch, cfg.align_last_k_layers)
        s_rep = get_proxy(s_h, attn).squeeze(1)

        L_supcon = supervised_contrastive_loss(s_rep, labels, cfg.temperature)
        L_sp = l2sp_penalty(student, init_state)

        loss = cfg.w_supcon * L_supcon + cfg.w_sp * L_sp

        loss.backward()
        nn.utils.clip_grad_norm_(trainable_params, cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        global_step += 1
        run_loss += float(loss.item())
        run_supcon += float(L_supcon.item())
        run_sp += float(L_sp.item())

        if (step + 1) % 25 == 0 or (step + 1) == len(train_loader):
            denom = step + 1
            print(
                f"[EP {ep+1}/{cfg.epochs} | {step+1}/{len(train_loader)} | gs {global_step}] "
                f"loss={run_loss/denom:.6f} "
                f"SUPCON={run_supcon/denom:.6f} "
                f"SP={run_sp/denom:.6f}"
            )

    final_epoch_avg_loss = run_loss / max(len(train_loader), 1)
    final_epoch_avg_supcon = run_supcon / max(len(train_loader), 1)
    final_epoch_avg_sp = run_sp / max(len(train_loader), 1)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 12) SAVE RAW MODEL
student.eval()
student.save_pretrained(OUT_DIR_RAW, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR_RAW)

# 13) CLEAN SAVE: RENAME gamma/beta -> weight/bias
def rename_layernorm_keys_in_state_dict(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    new_sd = {}
    for k, v in sd.items():
        nk = k.replace(".gamma", ".weight").replace(".beta", ".bias")
        new_sd[nk] = v.detach().cpu().clone()
    return new_sd

raw_state = student.state_dict()
clean_state = rename_layernorm_keys_in_state_dict(raw_state)

clean_model = AutoModelForMaskedLM.from_pretrained(MODEL_STEP3)
missing, unexpected = clean_model.load_state_dict(clean_state, strict=False)

print("\nCLEAN CHECKPOINT CHECK")
print("Missing keys   :", missing)
print("Unexpected keys:", unexpected)

clean_model.eval()
clean_model.save_pretrained(OUT_DIR_CLEAN, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR_CLEAN)

# 14) SAME-RUN RECHECK
# Recompute one batch loss to sanity-check same-run stability.
recheck_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn
)

recheck_loss = None
for batch in recheck_loader:
    batch = {k: v.to(DEVICE, non_blocking=False) for k, v in batch.items()}
    labels = batch["labels"]
    attn = batch["attention_mask"]

    with torch.no_grad():
        s_h = get_hidden_mean_lastk_grad(student, batch, cfg.align_last_k_layers)
        s_rep = get_proxy(s_h, attn).squeeze(1)
        L_supcon = supervised_contrastive_loss(s_rep, labels, cfg.temperature)
        L_sp = l2sp_penalty(student, init_state)
        recheck_loss = float((cfg.w_supcon * L_supcon + cfg.w_sp * L_sp).item())
    break

print("\nRECHECK one-batch loss:", recheck_loss)

# 15) SAVE REPORT
with open(os.path.join(OUT_DIR_CLEAN, "supcon_tinting_report.txt"), "w", encoding="utf-8") as f:
    f.write("STEP 3.5 — SUPERVISED CONTRASTIVE TINTING (CLEAN)\n")
    f.write(f"SEED = {SEED}\n")
    f.write(f"DEVICE = {DEVICE}\n")
    f.write(f"MODEL_STEP3 = {MODEL_STEP3}\n")
    f.write(f"DATA_PATH = {DATA_PATH}\n")
    f.write(f"OUT_DIR_RAW = {OUT_DIR_RAW}\n")
    f.write(f"OUT_DIR_CLEAN = {OUT_DIR_CLEAN}\n\n")
    f.write("CFG:\n")
    for k, v in cfg.__dict__.items():
        f.write(f"{k}: {v}\n")
    f.write("\nSplit info:\n")
    f.write(f"Train rows = {len(train_df)}\n")
    f.write(f"Val rows   = {len(val_df)} (not used)\n")
    f.write(f"Test rows  = {len(test_df)} (not used)\n")
    f.write("\nTraining summary:\n")
    f.write(f"total_steps = {total_steps}\n")
    f.write(f"warmup_steps = {warmup_steps}\n")
    f.write(f"final_epoch_avg_loss = {final_epoch_avg_loss}\n")
    f.write(f"final_epoch_avg_supcon = {final_epoch_avg_supcon}\n")
    f.write(f"final_epoch_avg_sp = {final_epoch_avg_sp}\n")
    f.write(f"recheck_one_batch_loss = {recheck_loss}\n")
    f.write(f"missing_keys = {missing}\n")
    f.write(f"unexpected_keys = {unexpected}\n")

with open(os.path.join(OUT_DIR_CLEAN, "step35_meta.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "seed": SEED,
            "device": DEVICE,
            "model_step3": MODEL_STEP3,
            "data_path": DATA_PATH,
            "out_dir_raw": OUT_DIR_RAW,
            "out_dir_clean": OUT_DIR_CLEAN,
            "train_rows_used": len(train_df),
            "config": cfg.__dict__,
            "total_steps": total_steps,
            "warmup_steps": warmup_steps,
            "final_epoch_avg_loss": final_epoch_avg_loss,
            "final_epoch_avg_supcon": final_epoch_avg_supcon,
            "final_epoch_avg_sp": final_epoch_avg_sp,
            "recheck_one_batch_loss": recheck_loss,
            "missing_keys": list(missing),
            "unexpected_keys": list(unexpected),
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nSTEP 3.5 SUPCON TINTING complete.")
print("Raw backbone   :", OUT_DIR_RAW)
print("Clean backbone :", OUT_DIR_CLEAN)

DEVICE: cuda
SEED  : 42
CFG(text_col='title_segmented', label_col='label', max_len=256, batch_size=16, epochs=1, lr=1e-05, weight_decay=0.01, warmup_ratio=0.06, grad_clip=1.0, freeze_embeddings=True, freeze_bottom_n_layers=9, freeze_mlm_head=True, align_last_k_layers=1, proxy='cls', mix_alpha=0.5, temperature=0.1, w_supcon=1.0, w_sp=0.0005)

FULL DATA LABEL DISTRIBUTION
label
0    187
1    249
2    569
Name: count, dtype: int64

STEP 3.5 USES TRAIN ONLY
Train: 703
Val  : 151 (not used)
Test : 151 (not used)

TRAIN LABEL DISTRIBUTION
label
0    131
1    174
2    398
Name: count, dtype: int64

Train dataset size: 703


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Freeze summary -> embeddings=True, bottom_layers=9/12, mlm_head=True
Trainable params: 21263616

TRAIN CONFIG
total_steps : 44
warmup_steps: 2
[EP 1/1 | 25/44 | gs 25] loss=2.835941 SUPCON=2.835941 SP=0.000000
[EP 1/1 | 44/44 | gs 44] loss=2.821901 SUPCON=2.821901 SP=0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


CLEAN CHECKPOINT CHECK
Missing keys   : []
Unexpected keys: []


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RECHECK one-batch loss: 2.7117397785186768

STEP 3.5 SUPCON TINTING complete.
Raw backbone   : /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step35_supcon_tinting_final_raw
Clean backbone : /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step35_supcon_tinting_final_clean


In [ ]:
# STEP 4 — FINAL FINE-TUNING ON DF5_UNSEEN (70/15/15)
!pip install -q transformers datasets scikit-learn sentencepiece

# 0) GLOBAL DETERMINISM
import os
import gc
import math
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore")

SEED = 42

def make_deterministic(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    random.seed(seed)
    np.random.seed(seed)
    set_seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("[INFO] deterministic algorithms not fully enabled:", e)

make_deterministic(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)
print("SEED   =", SEED)

# 1) PATHS & HYPERPARAMS
BASE_DIR = "/content/drive/MyDrive/Colab/myvietnam"

DATA_PATH = f"{BASE_DIR}/df5_moi_tachtu.csv"
MODEL_STEP35_CLEAN = f"{BASE_DIR}/cuoi_output_step35_supcon_tinting_final_clean"
TOKENIZER_C0 = f"{BASE_DIR}/C0_PhoBERT_base_v2"

OUTPUT_DIR = f"{BASE_DIR}/cuoi_output_step4_finetune_df5_from_step35_supcon_clean_repro"
FINAL_PATH = os.path.join(OUTPUT_DIR, "C6_final_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN       = 256
LR            = 1.5e-5
NUM_EPOCHS    = 5
TRAIN_BS      = 8
EVAL_BS       = 8
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06

# 2) LABEL MAP
label2id = {"positive": 0, "neutral": 1, "negative": 2}
id2label = {v: k for k, v in label2id.items()}

# 3) LOAD DATA
df5_unseen = pd.read_csv(DATA_PATH)

required_cols = ["title_segmented", "label"]
for col in required_cols:
    if col not in df5_unseen.columns:
        raise ValueError(f"Missing required column: {col}")

df5_unseen["title_segmented"] = df5_unseen["title_segmented"].astype(str).fillna("")

if df5_unseen["label"].dtype == object:
    unknown_labels = set(df5_unseen["label"].dropna().unique()) - set(label2id.keys())
    if unknown_labels:
        raise ValueError(f"Unknown labels found: {unknown_labels}")
    df5_unseen["label"] = df5_unseen["label"].map(label2id)

df5_unseen["label"] = df5_unseen["label"].astype(int)

invalid_labels = df5_unseen.loc[~df5_unseen["label"].isin([0, 1, 2])]
if len(invalid_labels) > 0:
    raise ValueError("Found invalid label values outside {0,1,2}")

print("\nFULL DATA LABEL DISTRIBUTION")
print(df5_unseen["label"].value_counts().sort_index())

# 4) SPLIT DATA (MUST MATCH STEP 3.5)
train_df, temp_df = train_test_split(
    df5_unseen,
    test_size=0.30,
    stratify=df5_unseen["label"],
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("\nDATA SPLIT")
print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))

print("\nTRAIN LABEL DISTRIBUTION")
print(train_df["label"].value_counts().sort_index())

print("\nVAL LABEL DISTRIBUTION")
print(val_df["label"].value_counts().sort_index())

print("\nTEST LABEL DISTRIBUTION")
print(test_df["label"].value_counts().sort_index())

# 5) HF DATASET
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# 6) TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_C0, use_fast=False)

def tokenize_fn(batch):
    return tokenizer(
        batch["title_segmented"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True, load_from_cache_file=False)
val_ds   = val_ds.map(tokenize_fn, batched=True, load_from_cache_file=False)
test_ds  = test_ds.map(tokenize_fn, batched=True, load_from_cache_file=False)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=keep_cols)
val_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)

# 7) LOAD MODEL FROM CLEAN STEP 3.5 CHECKPOINT
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_STEP35_CLEAN,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,   # only classifier head mismatch is expected
)

model.to(DEVICE)
print("\nLoaded clean backbone from:", MODEL_STEP35_CLEAN)

# 8) METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_each = f1_score(labels, preds, average=None, labels=[0, 1, 2])

    return {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_positive": float(f1_each[0]),
        "f1_neutral": float(f1_each[1]),
        "f1_negative": float(f1_each[2]),
    }

# 9) TRAINING ARGS
steps_per_epoch = math.ceil(len(train_ds) / TRAIN_BS)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

print("\nTRAINING CONFIG")
print("steps_per_epoch =", steps_per_epoch)
print("total_steps     =", total_steps)
print("warmup_steps    =", warmup_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,

    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    logging_strategy="steps",
    logging_steps=100,
    report_to="none",
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED,

    remove_unused_columns=False,
)

# 10) TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# 11) TRAIN
print("\n===== START TRAINING =====")
train_result = trainer.train()

print("\n===== TRAINING DONE =====")
print(train_result)

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric    :", trainer.state.best_metric)

# 12) TEST
print("\n===== EVALUATING TEST SET =====")
predictions = trainer.predict(test_ds)

preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

acc = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")
f1_each = f1_score(labels, preds, average=None, labels=[0, 1, 2])

print("\nFINAL TEST PERFORMANCE")
print("----------------------")
print("Accuracy     :", acc)
print("F1 Macro     :", f1_macro)
print("F1 Positive  :", f1_each[0])
print("F1 Neutral   :", f1_each[1])
print("F1 Negative  :", f1_each[2])

print("\nClassification Report")
print(classification_report(
    labels, preds,
    labels=[0, 1, 2],
    target_names=["positive", "neutral", "negative"],
    digits=4
))

print("\nConfusion Matrix")
print(confusion_matrix(labels, preds, labels=[0, 1, 2]))

# 13) SAVE FINAL MODEL
trainer.save_model(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)

with open(os.path.join(OUTPUT_DIR, "final_test_report.txt"), "w", encoding="utf-8") as f:
    f.write("STEP 4 — FINAL FINE-TUNING FROM CLEAN STEP 3.5 SUPCON TINTING\n")
    f.write(f"SEED = {SEED}\n")
    f.write(f"MODEL_STEP35_CLEAN = {MODEL_STEP35_CLEAN}\n")
    f.write(f"DATA_PATH = {DATA_PATH}\n")
    f.write(f"OUTPUT_DIR = {OUTPUT_DIR}\n")
    f.write(f"FINAL_PATH = {FINAL_PATH}\n\n")
    f.write(f"Accuracy = {acc:.6f}\n")
    f.write(f"F1 Macro = {f1_macro:.6f}\n")
    f.write(f"F1 Positive = {f1_each[0]:.6f}\n")
    f.write(f"F1 Neutral = {f1_each[1]:.6f}\n")
    f.write(f"F1 Negative = {f1_each[2]:.6f}\n")

print("\nFinal model saved at:", FINAL_PATH)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nDONE.")

DEVICE = cuda
SEED   = 42

FULL DATA LABEL DISTRIBUTION
label
0    187
1    249
2    569
Name: count, dtype: int64

DATA SPLIT
Train : 703
Val   : 151
Test  : 151

TRAIN LABEL DISTRIBUTION
label
0    131
1    174
2    398
Name: count, dtype: int64

VAL LABEL DISTRIBUTION
label
0    28
1    37
2    86
Name: count, dtype: int64

TEST LABEL DISTRIBUTION
label
0    28
1    38
2    85
Name: count, dtype: int64


Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Map:   0%|          | 0/151 [00:00<?, ? examples/s]

Map:   0%|          | 0/151 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== TRAINING DONE =====
TrainOutput(global_step=440, training_loss=0.42504917925054375, metrics={'train_runtime': 311.9939, 'train_samples_per_second': 11.266, 'train_steps_per_second': 1.41, 'total_flos': 462421831656960.0, 'train_loss': 0.42504917925054375, 'epoch': 5.0})

Best checkpoint: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step4_finetune_df5_from_step35_supcon_clean_repro/checkpoint-352
Best metric    : 0.845584315468272

===== EVALUATING TEST SET =====



FINAL TEST PERFORMANCE
----------------------
Accuracy     : 0.9006622516556292
F1 Macro     : 0.8756490841935541
F1 Positive  : 0.8620689655172413
F1 Neutral   : 0.8169014084507042
F1 Negative  : 0.9479768786127167

Classification Report
              precision    recall  f1-score   support

    positive     0.8333    0.8929    0.8621        28
     neutral     0.8788    0.7632    0.8169        38
    negative     0.9318    0.9647    0.9480        85

    accuracy                         0.9007       151
   macro avg     0.8813    0.8736    0.8756       151
weighted avg     0.9002    0.9007    0.8991       151


Confusion Matrix
[[25  2  1]
 [ 4 29  5]
 [ 1  2 82]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final model saved at: /content/drive/MyDrive/Colab/myvietnam/cuoi_output_step4_finetune_df5_from_step35_supcon_clean_repro/C6_final_model

DONE.


In [ ]:
#End